# SERENE AI — Phase 2: Text Emotion Model Training
## Fine-Tuning DistilRoBERTa on GoEmotions (28 Multi-Label Classes)

**Objective**:
- Fine-tune `distilroberta-base` on the curated GoEmotions benchmark dataset (54,263 comments).
- Preserve all 28 fine-grained emotion classes as multi-label targets.
- Mitigate class imbalance using `BCEWithLogitsLoss(pos_weight=...)`.
- Evaluate using Macro-F1, Micro-F1, per-class metrics, and Hamming loss.
- Export model checkpoint for the SERENE AI inference service.

In [ ]:
# 1. Environment Setup & GPU Verification
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn pandas tqdm

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss
from tqdm import tqdm

# Set seeds for exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")

In [ ]:
# 2. Load Dataset Manifest or Download Data Directly
BASE_URL = "https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data"
EMOTIONS_URL = f"{BASE_URL}/emotions.txt"

import urllib.request
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(EMOTIONS_URL, "data/emotions.txt")

with open("data/emotions.txt") as f:
    EMOTIONS = [l.strip() for l in f if l.strip()]
NUM_CLASSES = len(EMOTIONS)
print(f"Total Emotion Classes: {NUM_CLASSES}")

# Download benchmark splits if running standalone on Colab/Kaggle
splits = {}
for split in ["train", "dev", "test"]:
    url = f"{BASE_URL}/{split}.tsv"
    path = f"data/{split}.tsv"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    df = pd.read_csv(path, sep="\t", header=None, names=["text", "raw_labels", "id"])
    splits[split] = df
    print(f"{split.upper()} loaded: {len(df)} rows")

In [ ]:
# 3. Compute Positive Class Weights for BCEWithLogitsLoss
train_df = splits["train"]
pos_counts = np.zeros(NUM_CLASSES)
total_samples = len(train_df)

for lbl_str in train_df["raw_labels"]:
    indices = [int(x) for x in str(lbl_str).split(",") if x.strip()]
    for idx in indices:
        pos_counts[idx] += 1

neg_counts = total_samples - pos_counts
# Square root positive weighting with clamp to prevent gradient explosions
pos_weights = np.clip(np.sqrt(neg_counts / np.maximum(pos_counts, 1.0)), 1.0, 10.0)
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float).to(device)
print("Sample Positive Weights (First 5):", pos_weights[:5])

In [ ]:
# 4. PyTorch Dataset Definition
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")

class GoEmotionsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.labels = []
        for lbl in df["raw_labels"]:
            vec = np.zeros(NUM_CLASSES, dtype=np.float32)
            for idx in [int(x) for x in str(lbl).split(",") if x.strip()]:
                vec[idx] = 1.0
            self.labels.append(vec)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_ds = GoEmotionsDataset(splits["train"], tokenizer)
val_ds   = GoEmotionsDataset(splits["dev"], tokenizer)
test_ds  = GoEmotionsDataset(splits["test"], tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
# 5. Initialize Model & Training Loop
id2label = {i: name for i, name in enumerate(EMOTIONS)}
label2id = {name: i for i, name in enumerate(EMOTIONS)}

config = AutoConfig.from_pretrained(
    "distilbert/distilroberta-base",
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"
)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilroberta-base",
    config=config
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
epochs = 5
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
# 6. Evaluation Function
def evaluate(loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            logits = model(input_ids, attention_mask=attention_mask).logits
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= threshold).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    h_loss = hamming_loss(all_labels, all_preds)
    return macro_f1, micro_f1, h_loss

print("Ready for Training. Run training loop on GPU.")

In [ ]:
# 7. Training Loop with Model Checkpointing
best_macro_f1 = 0.0
SAVE_DIR = "checkpoints/text_model"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        with torch.cuda.amp.autocast():
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item()
    
    val_macro, val_micro, val_hloss = evaluate(val_loader)
    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f} | Val Macro-F1: {val_macro:.4f} | Val Micro-F1: {val_micro:.4f}")
    
    if val_macro > best_macro_f1:
        best_macro_f1 = val_macro
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"Saved Best Checkpoint to {SAVE_DIR} (Macro-F1: {best_macro_f1:.4f})")

# Final Test Evaluation
test_macro, test_micro, test_hloss = evaluate(test_loader)
print(f"Final Test Set Macro-F1: {test_macro:.4f} | Test Micro-F1: {test_micro:.4f} | Hamming Loss: {test_hloss:.4f}")